In [1]:
import time
import os
from datetime import datetime

import json
from tqdm.auto import tqdm
from dotenv import load_dotenv
import pandas as pd
from openai import OpenAI
import wikipediaapi
import wikipedia

In [2]:
load_dotenv()

True

In [3]:
wiki = wikipediaapi.Wikipedia(
    user_agent="MyFreshQABot/1.0 (contact: sucruba7@email.com)",
    language='ko',
    extract_format=wikipediaapi.ExtractFormat.WIKI
)

In [4]:
def safe_parse_json(text: str):
    text = text.strip()
    if text.startswith("```"):
        lines = text.splitlines()
        if lines and lines[0].startswith("```"):
            lines = lines[1:]
        if lines and lines[-1].startswith("```"):
            lines = lines[:-1]
        text = "\n".join(lines).strip()

    start_candidates = [p for p in (text.find("{"), text.find("[")) if p != -1]
    if not start_candidates:
        raise json.JSONDecodeError("No JSON found", text, 0)

    s = text[min(start_candidates):].lstrip()

    decoder = json.JSONDecoder()
    obj, end = decoder.raw_decode(s)
    return obj

### 규칙
- 제목 (분류)

In [6]:
wikipedia.set_lang('ko')
wiki_api = wikipediaapi.Wikipedia(
    language='ko',
    extract_format=wikipediaapi.ExtractFormat.WIKI,
    user_agent='MyApp/1.0 (contact@email.com)'
)

def get_wiki_text(keyword):
    """
    Wikipedia 텍스트 가져오기
    
    Args:
        keyword: 검색 키워드 (예: "임수정 배우", "2022 월드컵")
    
    Returns:
        str: Wikipedia 본문 텍스트 또는 None
    """
    try:
        search_results = wikipedia.search(keyword, results=3)
        
        if not search_results:
            return None
        
        for title in search_results:
            page = wiki_api.page(title)
            
            if page.exists():
                return page.text
        
        return None
    except Exception as e:
        return None

In [33]:
def get_wiki_text(keyword, max_length=10000):
    """
    Wikipedia 텍스트 가져오기 (길이 제한)
    
    Args:
        keyword: 검색 키워드
        max_length: 최대 문자 수 (기본 10,000자 ≈ 6,000 tokens)
    
    Returns:
        str: Wikipedia 본문 텍스트 또는 None
    """
    try:
        search_results = wikipedia.search(keyword, results=3)
        
        if not search_results:
            return None
        
        for title in search_results:
            page = wiki_api.page(title)
            
            if page.exists():
                text = page.text
                
                # 길이 제한 (앞부분 우선)
                if len(text) > max_length:
                    text = text[:max_length] + "\n\n[문서가 너무 길어 앞부분만 제공됨]"
                
                return text
        
        return None
        
    except Exception as e:
        return None

In [7]:
get_wiki_text("임수정 (배우)")

"임수정(1979년 7월 11일 ~ )은 대한민국의 배우이다.\n\n경력\n1998년 잡지 '쎄씨' 표지모델로 데뷔해 활동하기 시작했으며, 2001년 KBS의 시즌제 교육 드라마 《학교 4》를 통해 연기자로 데뷔하였다. 2003년 감독 김지운의 《장화, 홍련》에서 예민하면서도 서늘한 캐릭터인 소녀 수미를 맡아 2003년 심사위원 만장일치로 신인 여우주연상을 수상했다.\n2018년 영화 《당신의 부탁》에서 주연인 엄마 역할을 맡았다. 갑자기 나타난 아들과의 동거를 그린 이 영화는 제24회 브졸 국제아시아영화제 초청 및 넷팩 심사위원상을 수상하였고, 제16회 피렌체 한국영화제와 제6회 헬싱키 시네아시아에 초청되었다.\n\n출연 작품\n영화\n드라마\n예능\nTV\n라디오\n뮤직 비디오\n광고\n경력\n수상 및 후보\n각주\n외부 링크\n\n임수정 - 인스타그램\n임수정 - 한국영화 데이터베이스 \n(영어) 임수정 - 인터넷 영화 데이터베이스 \n(영어) 임수정 - 한시네마"

In [8]:
def build_message_keyword(system_prompt: str, user_prompt: str, today: str, question):
  return [
        {
            "role": "system",
            "content": system_prompt.format(TODAY=today),
        },
        {
            "role": "user",
            "content": user_prompt.format(QUESTION=question),
        },
    ]

def ask_solar_pro2_keyword(system_template: str, user_template: str, today: str, question: str):
  msg = build_message_keyword(system_template, user_template, today, question)
  res = client.chat.completions.create(
    model="solar-pro2",
    messages=msg,
    stream=False,
    temperature=0,
    top_p=1,
    reasoning_effort="high",
  )
  raw = res.choices[0].message.content.strip()
  return raw, safe_parse_json(raw)

In [ ]:
KEYWORD_PROMPT = """
당신은 위키피디아 검색용 키워드 추출 전문가입니다. 
사용자의 질문을 분석하여 위키피디아 검색에 가장 적합한 '문서 제목'을 결정하고, 반드시 아래 JSON 형식으로만 출력하세요.

### 규칙:
1. 반드시 순수 JSON만 출력하세요. (마크다운, 사족 금지)
2. `query` 키에 위키피디아 문서 제목을 담으세요.
3. 인물은 직업을 괄호 안에 명시합니다. (예: 임수정 (배우))
4. 사건/단체는 공식 명칭을 사용합니다. (예: 2022년 FIFA 월드컵)

### 출력 형식:
{{
  "query": "검색할 문서 제목"
}}

### 예시:
- 질문: "임수정 배우의 나이는?" -> {{"query": "임수정 (배우)"}}
- 질문: "카카오뱅크 26주 적금 금리가 뭐야?" -> {{"query": "카카오뱅크"}}
- 질문: "2022년 월드컵 우승국은?" -> {{"query": "2022년 FIFA 월드컵"}}
"""

USER_PROMPT_KEYWORD = """
### [입력]
QUESTION: {QUESTION}
"""

In [60]:
KEYWORD_PROMPT2 = """
당신은 위키피디아 검색 키워드 생성 전문가입니다.
질문을 분석하여 위키피디아 검색에 사용할 핵심 키워드를 추출하세요.

### [규칙]
1. **검색어 형식:** 문서 제목이 아닌, 검색 키워드입니다.
2. **간결하게:** 2-4단어 정도의 핵심어만 추출하세요.
3. **인물:** 이름 + 직업 (예: "임수정 배우")
4. **사건:** 핵심 명칭 (예: "2022 월드컵")
5. **단체/장소:** 고유명사만 (예: "카카오뱅크")

### [JSON 출력]
{{
  "query": "검색 키워드"
}}

### [예시]

질문: "임수정 배우의 나이는?"
출력: {{"query": "임수정 배우"}}

질문: "2022년 월드컵 우승국은?"
출력: {{"query": "2022 월드컵"}}

질문: "서울 인구는?"
출력: {{"query": "서울특별시"}}
"""

In [109]:
ROOT_DIR = "/data/ephemeral/pro-nlp-finalproject-nlp-13/data/"
dir = os.path.join(ROOT_DIR, 'ko-freshqa_2025_dev.csv')
df = pd.read_csv(dir, index_col=False)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 550 entries, 0 to 549
Data columns (total 20 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id              550 non-null    int64  
 1   split           550 non-null    object 
 2   domain          550 non-null    object 
 3   question        550 non-null    object 
 4   effective_year  550 non-null    object 
 5   next_review     283 non-null    object 
 6   false_premise   550 non-null    bool   
 7   num_hops        550 non-null    object 
 8   fact_type       550 non-null    object 
 9   source          550 non-null    object 
 10  answer_0        550 non-null    object 
 11  answer_1        550 non-null    object 
 12  answer_2        134 non-null    object 
 13  answer_3        11 non-null     object 
 14  answer_4        2 non-null      object 
 15  answer_5        0 non-null      float64
 16  answer_6        0 non-null      float64
 17  answer_7        0 non-null      flo

In [14]:
df_judge = pd.read_parquet('df_judge_partial.parquet')

In [15]:
TODAY = datetime.now().strftime("%Y-%m-%d")
client = OpenAI(
    api_key=os.getenv("UPSTAGE_API_KEY"),
    base_url="https://api.upstage.ai/v1"
)

In [16]:
question = df_judge['question'][2]
question = "배우 임수정이 몇 살인지 알려줘."

In [17]:
msg = build_message_keyword(KEYWORD_PROMPT, USER_PROMPT_KEYWORD, TODAY, question)
print(msg)

[{'role': 'system', 'content': '\n당신은 위키피디아 검색용 키워드 추출 전문가입니다. \n사용자의 질문을 분석하여 위키피디아 검색에 가장 적합한 \'문서 제목\'을 결정하고, 반드시 아래 JSON 형식으로만 출력하세요.\n\n### 규칙:\n1. 반드시 순수 JSON만 출력하세요. (마크다운, 사족 금지)\n2. `query` 키에 위키피디아 문서 제목을 담으세요.\n3. 인물은 직업을 괄호 안에 명시합니다. (예: 임수정 (배우))\n4. 사건/단체는 공식 명칭을 사용합니다. (예: 2022년 FIFA 월드컵)\n\n### 출력 형식:\n{\n  "query": "검색할 문서 제목"\n}\n\n### 예시:\n- 질문: "임수정 배우의 나이는?" -> {"query": "임수정 (배우)"}\n- 질문: "카카오뱅크 26주 적금 금리가 뭐야?" -> {"query": "카카오뱅크"}\n- 질문: "2022년 월드컵 우승국은?" -> {"query": "2022년 FIFA 월드컵"}\n'}, {'role': 'user', 'content': '\n### [입력]\nQUESTION: 배우 임수정이 몇 살인지 알려줘.\n'}]


In [19]:
a,b = ask_solar_pro2_keyword(KEYWORD_PROMPT, USER_PROMPT_KEYWORD, TODAY, question)
b

{'query': '임수정 (배우)'}

In [21]:
get_wiki_text(b['query'])

"임수정(1979년 7월 11일 ~ )은 대한민국의 배우이다.\n\n경력\n1998년 잡지 '쎄씨' 표지모델로 데뷔해 활동하기 시작했으며, 2001년 KBS의 시즌제 교육 드라마 《학교 4》를 통해 연기자로 데뷔하였다. 2003년 감독 김지운의 《장화, 홍련》에서 예민하면서도 서늘한 캐릭터인 소녀 수미를 맡아 2003년 심사위원 만장일치로 신인 여우주연상을 수상했다.\n2018년 영화 《당신의 부탁》에서 주연인 엄마 역할을 맡았다. 갑자기 나타난 아들과의 동거를 그린 이 영화는 제24회 브졸 국제아시아영화제 초청 및 넷팩 심사위원상을 수상하였고, 제16회 피렌체 한국영화제와 제6회 헬싱키 시네아시아에 초청되었다.\n\n출연 작품\n영화\n드라마\n예능\nTV\n라디오\n뮤직 비디오\n광고\n경력\n수상 및 후보\n각주\n외부 링크\n\n임수정 - 인스타그램\n임수정 - 한국영화 데이터베이스 \n(영어) 임수정 - 인터넷 영화 데이터베이스 \n(영어) 임수정 - 한시네마"

### A안 전략
- A -> 위키피디아에서만 검색 -> content + question -> 답 도출 => 최종 답


In [ ]:
SYSTEM_PROMPT_A1 = """
당신은 위키피디아 문서를 분석하여 질문에 답하는 전문가입니다.
제공된 긴 위키피디아 텍스트에서 질문과 관련된 정보를 찾아 답변하세요.

### [기준 정보]
- TODAY: {TODAY}

### [답변 절차]
1. **정보 탐색:** 위키 전문(제목, 본문, 섹션 포함)에서 질문 관련 문장 찾기
2. **사실 추출:** 핵심 정보(날짜, 이름, 수치) 추출
3. **계산 수행:** 날짜/나이/기간은 TODAY 기준 계산
4. **간결 답변:** 직접 답변만 작성 (20자 이내 권장)

### [중요 원칙]
- 위키 텍스트가 길어도 끝까지 스캔하세요
- **환각 금지:** 텍스트에 없는 내용은 절대 지어내지 마세요
- 정보 없으면 "정보 없음"이 정답입니다

### [JSON 출력]
{{
  "answer": "답변"
}}

### [예시1]
**위키 텍스트 (일부):**
"임수정(林秀晶, 1979년 9월 6일 ~ )은 대한민국의 배우이다. ... (후략 5000자)"

**질문:** "임수정 나이는?"
**출력:** {{"answer": "46세"}}

### [예시2]
**위키 텍스트 (일부):**
"서울특별시는 대한민국의 수도이다. ... 인구는 2023년 기준 약 960만 명이다. ... (후략)"

**질문:** "서울 인구는?"
**출력:** {{"answer": "약 960만 명"}}

### [예시3]
**위키 텍스트:**
(키 정보 언급 없음)

**질문:** "임수정 키는?"
**출력:** {{"answer": "정보 없음"}}
"""

USER_PROMPT_A1 = """
아래 위키피디아 문서를 읽고 질문에 답하세요.
텍스트가 길지만, 질문 관련 정보를 찾아 정확히 답변하세요.

### [위키피디아 전문]
{CONTENT}

### [질문]
{QUESTION}
"""

In [38]:
SYSTEM_PROMPT_A2 = """
당신은 위키피디아 문서를 분석하여 질문에 답하는 전문가입니다.
제공된 위키피디아 텍스트에서 질문 관련 정보를 찾아 답변하세요.

### [기준 정보]
- TODAY: {TODAY}

### [답변 원칙]
1. **위키 우선:** 제공된 위키 텍스트에 답이 있으면 그것을 사용
2. **지식 사용 제한:** 위키 텍스트가 없거나 답이 명시되지 않으면, 확실히 아는 범위에서만 답변 (불확실하면 "정보 없음")
3. **계산 수행:** 날짜/나이/기간은 TODAY 기준 계산
4. **불필요한 부연 금지:** 질문에 대한 짧은 정답(숫자/명사구/한 문장)만 출력

### [중요]
- 위키 텍스트 길어도 끝까지 스캔
- 환각 금지: 모르면 "정보 없음"
- 부연 설명 금지

### [JSON 출력]
{{
  "answer": "답변"
}}

### [예시]

**위키:** "대한민국(1948년 8월 15일 정부 수립)..."
**질문:** "대한민국 건국일?"
**출력:** {{"answer": "1948년 8월 15일"}}

**위키:** "파이썬은 1991년 귀도 반 로섬이 개발..."
**질문:** "파이썬 개발자?"
**출력:** {{"answer": "귀도 반 로섬"}}

**위키:** (없음)
**질문:** "지구 둘레는?"
**출력:** {{"answer": "약 4만 km"}}

**위키:** (나이 정보 없음)
**질문:** "BTS RM 나이?"
**출력:** {{"answer": "정보 없음"}}
"""

USER_PROMPT_A2 = """
### [위키피디아 전문]
{CONTENT}

### [질문]
{QUESTION}

위키에 답이 있으면 사용하고, 없거나 불완전하면 당신의 지식으로 답변하세요.
"""

In [39]:
def build_message_A(system_prompt: str, user_prompt: str, today: str, question: str, content: str):
      return [
        {
            "role": "system",
            "content": system_prompt.format(TODAY=today),
        },
        {
            "role": "user",
            "content": user_prompt.format(QUESTION=question, CONTENT=content),
        },
    ]

def ask_solar_pro2_A(system_template: str, user_template: str, today: str, question: str, content: str):
  msg = build_message_A(system_template, user_template, today, question, content=content)
  
  res = client.chat.completions.create(
    model="solar-pro2",
    messages=msg,
    stream=False,
    temperature=0,
    top_p=1,
    reasoning_effort="high"
  )

  raw = res.choices[0].message.content.strip()

  return raw, safe_parse_json(raw)

In [62]:
sample_size = 30
df_judge_a = df_judge[df_judge['model_answer_type'] == "A"]
sample_df = df_judge_a.sample(sample_size, random_state=42)

In [41]:
for i, (idx, row) in tqdm(enumerate(sample_df.iterrows(), 1), 
                         total=sample_size, 
                         desc="Question Sampling"):
    print(f"{i}. [ID: {row.get('id', 'N/A')}]")
    
    q = row['question']
    
    try:
        _, keyword_json = ask_solar_pro2_keyword(KEYWORD_PROMPT, USER_PROMPT_KEYWORD, TODAY, q)
        keyword = keyword_json.get('query', q)
        wiki_content = get_wiki_text(keyword)
        
        if wiki_content:
            print(f"[Wiki Data 로드 성공 (텍스트 길이]: {len(wiki_content)}자)")
        if not wiki_content:
            wiki_content = "관련 정보 없음"

        raw_text, parsed_json = ask_solar_pro2_A(
            SYSTEM_PROMPT_A2, 
            USER_PROMPT_A2, 
            TODAY, 
            q,          
            wiki_content
        )
        
    except Exception as e:
        print(f"[Error]: {e}")
        parsed_json = {"answer": "에러 발생"}

    print(f"[Question]: {q}")
    print(f"[Keyword]: {keyword}")
    print(f"[실제 정답]: {row.get('answer_0', '')}")
    print(f"[Model 답]: {parsed_json.get('answer', '')}")
    print("=" * 50)

Question Sampling:   0%|          | 0/30 [00:00<?, ?it/s]

1. [ID: 3077]
[Wiki Data 로드 성공 (텍스트 길이]: 10022자)
[Question]: 대한항공은 언제 창립되었지?
[Keyword]: 대한항공
[실제 정답]: 1969년 3월 1일
[Model 답]: 1969년 3월 1일
2. [ID: 3070]
[Wiki Data 로드 성공 (텍스트 길이]: 10022자)
[Question]: 비틀즈의 주요 음악 장르는 뭐야?
[Keyword]: 비틀즈
[실제 정답]: 록(Rock)
[Model 답]: 록, 팝, 로큰롤, 발라드, 사이키델릭 록, 하드 록
3. [ID: 3210]
[Wiki Data 로드 성공 (텍스트 길이]: 3092자)
[Question]: 산업혁명이 처음으로 시작된 나라가 어디야?
[Keyword]: 산업혁명
[실제 정답]: 영국
[Model 답]: 영국
4. [ID: 3166]
[Wiki Data 로드 성공 (텍스트 길이]: 10022자)
[Question]: 엘리자베스 2세는 언제 왕위에 올랐나요?
[Keyword]: 엘리자베스 2세 (영국 여왕)
[실제 정답]: 1952년
[Model 답]: 1952년 2월 6일
5. [ID: 3148]
[Wiki Data 로드 성공 (텍스트 길이]: 10022자)
[Question]: 세리에 A의 22-23 시즌 우승팀은 어디인가요?
[Keyword]: 2022-23 세리에 A
[실제 정답]: SSC 나폴리
[Model 답]: 나폴리
6. [ID: 3122]
[Wiki Data 로드 성공 (텍스트 길이]: 1822자)
[Question]: 임재범의 3집 타이틀곡이 뭐지?
[Keyword]: 이제 너없이는 못살아
[실제 정답]: 고해
[Model 답]: 정보 없음
7. [ID: 3246]
[Wiki Data 로드 성공 (텍스트 길이]: 1866자)
[Question]: 내부구조가 없어 더 작은 그 무엇으로 분리될수 없는 궁극적인 기본 입자를 무엇이라고 하나요?
[Keyword]: 기본 입자
[실제 정답]: 쿼크
[Model 답]: 기본 입자


In [63]:
for i, (idx, row) in tqdm(enumerate(sample_df.iterrows(), 1), 
                         total=sample_size, 
                         desc="Question Sampling"):
    print(f"{i}. [ID: {row.get('id', 'N/A')}]")
    
    q = row['question']
    
    try:
        _, keyword_json = ask_solar_pro2_keyword(KEYWORD_PROMPT2, USER_PROMPT_KEYWORD, TODAY, q)
        keyword = keyword_json.get('query', q)
        wiki_content = get_wiki_text(keyword)
        
        if wiki_content:
            print(f"[Wiki Data 로드 성공 (텍스트 길이]: {len(wiki_content)}자)")
        if not wiki_content:
            wiki_content = "관련 정보 없음"

        raw_text, parsed_json = ask_solar_pro2_A(
            SYSTEM_PROMPT_A2, 
            USER_PROMPT_A2, 
            TODAY, 
            q,          
            wiki_content
        )
        
    except Exception as e:
        print(f"[Error]: {e}")
        parsed_json = {"answer": "에러 발생"}

    print(f"[Question]: {q}")
    print(f"[Keyword]: {keyword}")
    print(f"[실제 정답]: {row.get('answer_0', '')}")
    print(f"[Model 답]: {parsed_json.get('answer', '')}")
    print("=" * 50)

Question Sampling:   0%|          | 0/30 [00:00<?, ?it/s]

1. [ID: 3077]
[Wiki Data 로드 성공 (텍스트 길이]: 10022자)
[Question]: 대한항공은 언제 창립되었지?
[Keyword]: 대한항공
[실제 정답]: 1969년 3월 1일
[Model 답]: 1969년 3월 1일
2. [ID: 3070]
[Wiki Data 로드 성공 (텍스트 길이]: 10022자)
[Question]: 비틀즈의 주요 음악 장르는 뭐야?
[Keyword]: 비틀즈
[실제 정답]: 록(Rock)
[Model 답]: 록, 팝, 로큰롤, 비트, 발라드, 인도 음악, 사이키델릭, 하드 록
3. [ID: 3210]
[Wiki Data 로드 성공 (텍스트 길이]: 3092자)
[Question]: 산업혁명이 처음으로 시작된 나라가 어디야?
[Keyword]: 산업혁명 시작
[실제 정답]: 영국
[Model 답]: 영국
4. [ID: 3166]
[Wiki Data 로드 성공 (텍스트 길이]: 10022자)
[Question]: 엘리자베스 2세는 언제 왕위에 올랐나요?
[Keyword]: 엘리자베스 2세 여왕
[실제 정답]: 1952년
[Model 답]: 1952년 2월 6일
5. [ID: 3148]
[Wiki Data 로드 성공 (텍스트 길이]: 1170자)
[Question]: 세리에 A의 22-23 시즌 우승팀은 어디인가요?
[Keyword]: 세리에 A 22-23 시즌
[실제 정답]: SSC 나폴리
[Model 답]: 나폴리
6. [ID: 3122]
[Wiki Data 로드 성공 (텍스트 길이]: 10022자)
[Question]: 임재범의 3집 타이틀곡이 뭐지?
[Keyword]: 임재범 3집
[실제 정답]: 고해
[Model 답]: 고해
7. [ID: 3246]
[Wiki Data 로드 성공 (텍스트 길이]: 1445자)
[Question]: 내부구조가 없어 더 작은 그 무엇으로 분리될수 없는 궁극적인 기본 입자를 무엇이라고 하나요?
[Keyword]: 소립자
[실제 정답]: 쿼크
[Model 답]: 기본입자
8. 

In [ ]:

# wikipedia -> 답 + 근거
# tavily -> url n개 -> text -> 답 + 근거
# wikiepdia -> 답 + 근거 -> 관련 없음 -> tavily




